# 6 自定义 matmul 算子接入 Qwen3-8B

上一章完成了自定义量化算子QmmCustom的开发、编译部署和测试验证。本章会将该自定义算子接入 Qwen3-8B 模型，并测试其在模型中的性能。

本章大纲如下：
- 环境准备
- QmmCustom 算子部署确认
- 模型权重量化并进行量化模型推理
- 自定义算子接入模型
- 模型性能测试
    - **任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程**
---

## 1 环境准备

准备运行目录，把 CANN 环境变量导入当前 Notebook。

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_repo_root():
    repo_roots = []
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'reference_practice/model_inference_optimization/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'reference_practice' / 'model_inference_optimization' / 'qwen3_8b'
os.environ['TUTORIAL_DIR'] = str(TUTORIAL_DIR)
SRC_DIR = TUTORIAL_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment.')
print('CANN包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

RECIPE_ROOT = TUTORIAL_DIR / 'src/inference_scripts/recipe_qwen3_8b'
QMM_CUSTOM_DIR = SRC_DIR / 'op_custom' / 'qmm_custom'
QMM_EXT_DIR = SRC_DIR / 'op_custom' / 'torch_ops_extension'

print('教程目录 =', TUTORIAL_DIR)
os.environ['RECIPE_ROOT'] = str(RECIPE_ROOT)
print('推理模型目录 =', RECIPE_ROOT)
print('算子工程目录 =', QMM_CUSTOM_DIR)
print('PyTorch扩展目录 =', QMM_EXT_DIR)
print('[环境准备] 完成', flush=True)

安装所需的依赖：

In [ ]:
%pip install -r {RECIPE_ROOT}/models/qwen/requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn

---
## 2 QmmCustom 算子部署确认

确认上一章编译部署的自定义算子包已正确安装。如未安装，请在运行以下单元前先完成第5章的编译部署步骤，包括编译安装自定义算子包与C++扩展。

In [ ]:
# 确认自定义算子包存在
import os
customize_path = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'opp/vendors/customize'
print(customize_path)
if customize_path.exists() and (customize_path / 'op_api/lib/libcust_opapi.so').exists():
    print(f'自定义算子包已安装: {customize_path}')
else:
    print('警告: 自定义算子包未找到，请先完成第5章的编译部署。')


In [ ]:
# 如未安装，可取消下面两行的注释，在此处重新执行自定义算子包编译安装
#!cd {QMM_CUSTOM_DIR} && bash build.sh
#!cd {QMM_CUSTOM_DIR}/build_out && chmod +x *.run && ./*.run

## 3 模型权重量化并进行量化模型推理

自定义算子 QmmCustom 是一个 INT8 量化 matmul 算子，需要在 W8A8（Weight INT8 + Activation INT8）量化模型上才能发挥作用。本节使用昇腾量化工具 [AMCT(Ascend Model Compression Toolkit)](https://gitcode.com/cann/amct) 将已有的 Qwen3-8B BF16 权重导出为 W8A8 INT8 量化权重，然后用 recipes 框架跑通量化模型推理，为下一节的自定义算子接入提供量化模型基线。


### 3.1 使用 AMCT 导出 W8A8 量化权重

W8A8 INT8 量化将模型权重和激活值从 BF16/FP32 压缩为 INT8，减少内存占用和计算量。昇腾官方大模型量化工具 AMCT 支持通过命令行直接导出 Qwen3-8B W8A8 部署权重，输出目录为 HuggingFace 风格的 compressed-tensors 目录结构，recipes 框架可直接加载。


本章需复用前置章节的非量化模型权重，如环境中已有可跳过下方下载步骤；如环境中没有，需**取消下方code cell中的注释**进行下载，如下下载命令会将非量化模型的权重下载到 local-dir 指定的路径下，即{TUTORIAL_DIR}/src/data/models/Qwen3-8B。

In [ ]:
#!export HF_ENDPOINT=https://hf-mirror.com && hf download Qwen/Qwen3-8B --local-dir {TUTORIAL_DIR}/src/data/models/Qwen3-8B

如环境中已准备 W8A8 量化权重，可在下方章节通过 `W8A8_OUTPUT` 直接指定路径，跳过下方量化步骤；     
如**未准备需取消下方code cell中的注释**进行权重量化，使用 AMCT 通过命令行直接导出 Qwen3-8B W8A8 INT8 部署权重。流程主要包含如下三步：1、获取 AMCT 源码；2、安装 AMCT 依赖；3、导出 Qwen3-8B W8A8 部署权重。其中`--model`指定的是非量化模型权重的路径，`--output_dir`指定的是导出的量化模型权重的路径。

In [ ]:
"""
import os
import subprocess
import sys
from pathlib import Path

print('[AMCT 量化] 开始')

# 克隆 AMCT 源码
AMCT_DIR = Path(os.environ['TUTORIAL_DIR']) / 'src/amct'
if not AMCT_DIR.exists():
    print('[AMCT] 克隆源码...')
    subprocess.check_call(['git', 'clone', 'https://gitcode.com/cann/amct.git', str(AMCT_DIR)])

# 安装 AMCT 依赖
print('[AMCT] 安装依赖...')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-r', str(AMCT_DIR / 'requirements.txt'),
    '-i', 'https://pypi.tuna.tsinghua.edu.cn/simple',
    '--trusted-host', 'pypi.tuna.tsinghua.edu.cn',
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', 'setuptools<82',
    '-i', 'https://pypi.tuna.tsinghua.edu.cn/simple',
    '--trusted-host', 'pypi.tuna.tsinghua.edu.cn',
])

# 执行 W8A8 量化导出
print('[AMCT] 执行量化导出...')
W8A8_OUTPUT = Path(os.environ['TUTORIAL_DIR']) / 'src/data/models' / 'Qwen3-8B-W8A8'
deploy_env = os.environ.copy()
deploy_env['PYTHONPATH'] = str(AMCT_DIR)
subprocess.check_call([
    sys.executable, '-m', 'amct_pytorch.deploy',
    '--model', Path(os.environ['TUTORIAL_DIR']) / 'src/data/models' / 'Qwen3-8B',
    '--model_name', 'qwen3',
    '--granularity', 'block',
    '--quant_target', 'attn-linear', 'mlp',
    '--quant_dtype', 'int',
    '--bit_config', 'amct_pytorch/configs/w8a8.yaml',
    '--output_dir', str(W8A8_OUTPUT),
], cwd=str(AMCT_DIR), env=deploy_env)

print(f'[AMCT 量化] 完成，W8A8 权重目录 = {W8A8_OUTPUT}')

print('[注意] AMCT 可能降级 torch_npu 版本，如后续推理报错请重装 torch_npu wheel，即重新执行环境准备章节的第二个code cell。')
"""

### 3.2 准备 W8A8 推理 YAML

W8A8 量化推理使用与 BF16 推理相同的 recipes 框架，但需要使用 W8A8 量化模型路径和适配的 YAML 配置。量化配置由权重目录 `config.json` 内嵌的 `quantization_config` 自动加载，YAML 无需显式声明量化参数。

通过如下操作将 YAML 配置中的模型路径指定为量化权重路径 `W8A8_OUTPUT`：

In [ ]:
from inference_scripts.recipe_workflow import prepare_runtime_yaml

W8A8_OUTPUT = TUTORIAL_DIR / 'src/data/models/Qwen3-8B-W8A8'

YAML_TEMPLATE =  RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_1tp.yaml'
YAML_A8W8  = YAML_TEMPLATE

prepare_runtime_yaml(YAML_TEMPLATE, YAML_A8W8, model_path=str(W8A8_OUTPUT))

### 3.3 执行量化推理

调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，确认量化推理正常运行。

In [ ]:
import sys, subprocess, os

env = os.environ.copy()
python_dir = os.path.dirname(sys.executable)
env["PATH"] = python_dir + ":" + env.get("PATH", "")
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

---
## 4 自定义算子接入模型

### 4.1 定位替换点

Qwen3-8B 推理框架中的 W8A8 量化 matmul 通过 `CompressedTensorsW8A8Int8LinearMethod` 类实现，其 `apply_weights` 方法中使用 `torch_npu.npu_quant_matmul` 执行 INT8 量化矩阵乘法。我们需要将这个调用替换为自定义的 `torch.ops.custom.qmm_custom` 算子。

查看原始实现：

In [ ]:
!grep -n 'npu_quant_matmul' {SRC_DIR}/inference_scripts/recipe_qwen3_8b/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py

### 4.2 替换实现

在 `CompressedTensorsW8A8Int8LinearMethod` 中，将 `torch_npu.npu_quant_matmul` 替换为 `torch.ops.custom.qmm_custom`。

替换后的方法为：

```python
import custom_ops

def apply_weights(self, layer: torch.nn.Module,
                x: torch.Tensor,
                bias: Optional[torch.Tensor],
                dynamic_scale: Optional = None,
                out_dtype: Optional = torch.bfloat16
                ) -> Union[torch.Tensor, Dict[str, Any]]:
    if dynamic_scale is not None:
        x_scale = dynamic_scale
    else:
        x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)
    out_shape_dim_0 = x.size()[:-1]
    x = x.view(-1, x.size(-1))
    x_scale = x_scale.view(-1)

    # --- 替换核心: npu_quant_matmul → qmm_custom ---
    x = torch.ops.custom.qmm_custom(x, layer.weight,
                                layer.weight_scale.view(-1),
                                pertoken_scale=None if out_dtype == torch.int32 else x_scale,
                                dtype=0 if out_dtype == torch.int32 else 1)
    # --- 替换结束 ---

    x = x.view(out_shape_dim_0 + (-1, ))
    if out_dtype == torch.int32:
        return x, x_scale
    else:
        return x
```

执行如下操作替换推理网络中调用的接口实现：

In [ ]:
# 4.2 替换实现: 将 torch_npu.npu_quant_matmul 替换为 torch.ops.custom.qmm_custom

# 1: 导入custom_ops
CT_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
ct_code = CT_FILE.read_text(encoding='utf-8')

if 'import custom_ops' not in ct_code:
    # 在 'import torch' 行后插入 'import custom_ops'
    ct_code = ct_code.replace('import torch\n', 'import torch\nimport custom_ops\n')
    CT_FILE.write_text(ct_code, encoding='utf-8')
    print('[插入] compressed_tensors.py: 已添加 import custom_ops')
else:
    print('[插入] compressed_tensors.py: import custom_ops 已存在，跳过')

# 2: 在 compressed_tensors_w8a8_int8.py 中替换 npu_quant_matmul → qmm_custom
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
print('[替换] 目标文件:', W8A8_FILE)

code = W8A8_FILE.read_text(encoding='utf-8')

# 替换 npu_quant_matmul 调用为 qmm_custom 调用
old_call = (
    'x = torch_npu.npu_quant_matmul(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    bias=layer.bias,\n'
    '                                    output_dtype=out_dtype)'
)
new_call = (
    '# --- 替换核心: npu_quant_matmul → qmm_custom ---\n'
    '        x = torch.ops.custom.qmm_custom(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    dtype=0 if out_dtype == torch.int32 else 1)\n'
    '        # --- 替换结束 ---'
)

if old_call in code:
    code = code.replace(old_call, new_call)
    W8A8_FILE.write_text(code, encoding='utf-8')
    print('[替换] 成功: npu_quant_matmul → qmm_custom')
else:
    print('[替换] 文件中未找到 npu_quant_matmul 调用，可能已替换')

# 验证替换结果
result = W8A8_FILE.read_text(encoding='utf-8')
if 'torch.ops.custom.qmm_custom' in result:
    print('[验证] 替换已生效，当前使用 qmm_custom 算子')
else:
    print('[验证] 替换未生效，请检查文件内容')


### 4.3 替换实现后执行量化推理

调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，确认对比替换前的生成文本是否一致。

In [ ]:
import sys, subprocess, os

env = os.environ.copy()
python_dir = os.path.dirname(sys.executable)
env["PATH"] = python_dir + ":" + env.get("PATH", "")
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

---
## 5 模型性能测试

### 任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程

将 QmmCustom 算子接入模型后，使用 Profiling 工具观测模型推理性能，对比自定义算子替换前后的 matmul 耗时变化。

### 5.1 准备 Profiling YAML

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将 YAML 配置中的enable_profiler配置为 True ：

In [ ]:
from inference_scripts.recipe_workflow import prepare_profiling_yaml

YAML_TEMPLATE = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_1tp.yaml'
YAML_A8W8_PROFILER  = YAML_TEMPLATE

prepare_profiling_yaml(YAML_TEMPLATE, YAML_A8W8_PROFILER, enable_profiler=True)

### 5.2 启动 Profiling 推理

使用 recipes 命令启动带 Profiler 的推理，推理结束后会在结果目录中生成 `prof/` 产物。

In [ ]:
RES_DIR = RECIPE_ROOT / 'models/qwen/res'
!rm -rf RES_DIR

import sys, subprocess, os

env = os.environ.copy()
python_dir = os.path.dirname(sys.executable)
env["PATH"] = python_dir + ":" + env.get("PATH", "")
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_PROFILER],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

### 5.3 查看 Profiling 结果

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv`中展示了模型调用的所有算子的统计信息，可快速确认自定义算子 QmmCustom 的调用情况。`kernel_details.csv`中展示了详细的接口调用信息，可自行查看筛选自己关注的算子的性能。

通过如下操作统计 `kernel_details.csv` 中的 QmmCustom 算子耗时，可自行对比自定义算子替换前后的性能差异。

In [ ]:
import pandas as pd
from pathlib import Path

RES_DIR = RECIPE_ROOT / 'models/qwen/res'

# 取最新的 profiling 结果目录
result_dirs = sorted([d for d in RES_DIR.iterdir() if d.is_dir()], key=lambda d: d.name, reverse=True)
latest_result = result_dirs[0]
print(f'最新结果目录: {latest_result}')

def find_kernel_csv(phase_dir):
    matches = list(phase_dir.rglob('kernel_details.csv'))
    return matches[0] if matches else None

prefill_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_1tp/prof/prefill')
decode_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_1tp/prof/decode')
assert prefill_csv and decode_csv, 'kernel_details.csv 未找到，请确认 profiling 已完成'
print(f'Prefill CSV: {prefill_csv}')
print(f'Decode CSV: {decode_csv}')

# 读取 CSV 并筛选 QmmCustom 算子
prefill_df = pd.read_csv(prefill_csv)
decode_df = pd.read_csv(decode_csv)

prefill_qmm = prefill_df[prefill_df['Type'] == 'QmmCustom'].copy()
decode_qmm = decode_df[decode_df['Type'] == 'QmmCustom'].copy()

# 按 Input Shapes 分组，统计每种 shape 下的平均 Duration
def qmm_duration_stats(df, phase):
    stats = df.groupby(['Input Shapes', 'Output Data Types'])['Duration(us)'].agg(['mean', 'count']).reset_index()
    stats.columns = ['Input Shapes', 'Output Dtype', 'Avg Duration(us)', 'Calls']
    stats = stats.sort_values('Avg Duration(us)', ascending=False).reset_index(drop=True)
    stats.insert(0, 'Phase', phase)
    return stats

prefill_stats = qmm_duration_stats(prefill_qmm, 'Prefill')
decode_stats = qmm_duration_stats(decode_qmm, 'Decode')

combined = pd.concat([prefill_stats, decode_stats], ignore_index=True)
display(combined.style.set_caption('QmmCustom 各 Shape 平均耗时 (us)'))

---

### 小结

本章完成了 QmmCustom 算子接入 Qwen3-8B 模型，并通过 Profiling 获取模型性能。